# Badger_vision — Google Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dillun-Holmes/AI_vision_model/blob/main/notebooks/colab_quickstart.ipynb)

> **Tip:** Go to *Runtime > Change runtime type* and select **GPU** (T4 or better).

### Supported Datasets
| Source | Layout |
|--------|--------|
| Badger Factory – Object Detection | `yolo/images/{train,val}.7z` + `labels/{train,val}.7z` |
| Badger Factory – Keypoints | Same YOLO layout |
| Badger Factory – Classification | `classifier/evolving_ds_*/{train,val}.7z` |
| COCO Export | `{train,val}.7z` with `coco_instances.json` |
| Plain YOLO / COCO | Already extracted on disk |
| No dataset | Runs with synthetic demo data |

Archives: `.zip` `.7z` `.tar.gz` `.tar.bz2` `.tar.xz` `.rar`

## 1 — Install

In [ ]:
!pip install -q https://github.com/Dillun-Holmes/BadgerviAI_releases/releases/download/v4.3.0/badger_vision-4.3.0-py3-none-any.whl
!pip install -q tqdm py7zr rarfile

## 2 — GPU Check

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime > Change runtime type > GPU"
)
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu_name}  ({gpu_mem:.1f} GB)")
print(f"CUDA : {torch.version.cuda}  |  PyTorch: {torch.__version__}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda:0")

## 3 — Configure Training

Settings are loaded from `train_config.yaml`. Edit that file to change dataset, task, epochs, etc.
You can also override values directly in the cell below.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  Configuration — loaded from train_config.yaml            ║
# ╚════════════════════════════════════════════════════════════╝
# Edit train_config.yaml to change settings, or override below.

import yaml
from pathlib import Path

_cfg_path = Path("train_config.yaml")
if not _cfg_path.exists():
    _cfg_path = Path(__file__).resolve().parent / "train_config.yaml" if "__file__" in dir() else Path("train_config.yaml")

_tcfg = {}
if _cfg_path.exists():
    with open(_cfg_path) as _f:
        _tcfg = yaml.safe_load(_f) or {}
    print(f"Loaded config from {_cfg_path}")
else:
    print("train_config.yaml not found — using defaults")

DATASET_PATH = _tcfg.get("dataset_path", "")
TASK         = _tcfg.get("task", "detection")
EPOCHS       = _tcfg.get("epochs", 100)
BATCH_SIZE   = _tcfg.get("batch_size", 0)
IMG_SIZE     = _tcfg.get("img_size", 640)
LR           = _tcfg.get("lr", 0.01)
MODEL        = _tcfg.get("model", "resnext")

## 4 — Prepare Dataset

Auto-detects format, extracts archives, converts YOLO to COCO if needed.

In [ ]:
import importlib.util
import json
import shutil
import textwrap
from pathlib import Path

import numpy as np
from PIL import Image

# Load helpers from linux_train.py
REPO_DIR = Path("/content/AI_vision_model")
if not REPO_DIR.exists():
    import subprocess
    subprocess.check_call(
        ["git", "clone", "--depth=1",
         "https://github.com/Dillun-Holmes/AI_vision_model.git",
         str(REPO_DIR)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
spec = importlib.util.spec_from_file_location("linux_train", str(REPO_DIR / "notebooks" / "linux_train.py"))
lt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lt)

WORKSPACE = Path("/content/badger_vision_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)

USE_REAL_DATA = bool(DATASET_PATH) and Path(DATASET_PATH).exists()

if USE_REAL_DATA:
    dataset_path = Path(DATASET_PATH).resolve()
    if dataset_path.is_file() and lt.is_archive(dataset_path):
        extract_dest = WORKSPACE / "dataset"
        if extract_dest.exists():
            shutil.rmtree(extract_dest)
        dataset_root = lt.extract_archive(dataset_path, extract_dest)
    else:
        dataset_root = dataset_path

    dataset_root = lt.resolve_dataset_root(dataset_root, TASK)
    fmt = lt.detect_format(dataset_root)
    print(f"Detected format: {fmt}")

    if fmt == "badger_yolo":
        cls = dataset_root / "classes.txt"
        data_info = lt.prepare_badger_yolo(dataset_root, WORKSPACE, cls if cls.exists() else None)
    elif fmt == "badger_classifier":
        data_info = lt.prepare_badger_classifier(dataset_root, WORKSPACE)
    elif fmt == "coco_archive":
        data_info = lt.prepare_coco_archive(dataset_root, WORKSPACE)
    elif fmt == "yolo_flat":
        data_info = lt.prepare_yolo_flat(dataset_root, WORKSPACE)
    elif fmt == "coco_flat":
        data_info = lt.prepare_coco_flat(dataset_root, WORKSPACE)
    else:
        raise RuntimeError(f"Unknown format in {dataset_root}")

    num_classes = lt.detect_num_classes(data_info["train_ann_file"])
    num_train = lt.count_images(data_info["train_ann_file"])
    print(f"Classes: {num_classes}  |  Train images: {num_train}")
else:
    # Synthetic demo
    print("No DATASET_PATH — creating synthetic demo data.")
    ROOT = Path("/content/badger_vision_demo")
    CONFIGS = ROOT / "configs"
    DATA = ROOT / "data"
    IMGS = DATA / "images"
    CONFIGS.mkdir(parents=True, exist_ok=True)
    IMGS.mkdir(parents=True, exist_ok=True)

    (CONFIGS / "resnext_nano.yaml").write_text(textwrap.dedent("""model:\n  name: BadgerResNeXt-Nano\n  type: resnext\n  backbone: resnext_nano\n  neck_channels: 64\n  head_depth: 2\n  use_ghost: true\n  num_classes: 80\n  image_size: 640\ntraining:\n  epochs: 3\n  base_lr: 0.01\n  warmup_epochs: 1\n  accumulation_steps: 1\n  early_stopping_patience: 30\n"))
    coco = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "object"}]}
    ann_id = 0
    for i in range(4):
        fname = f"img_{i:04d}.jpg"
        img = Image.fromarray(np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8))
        img.save(str(IMGS / fname))
        coco["images"].append({"id": i, "file_name": fname, "width": 640, "height": 640})
        for _ in range(2):
            x, y = int(np.random.randint(0, 400)), int(np.random.randint(0, 400))
            w, h = int(np.random.randint(30, 200)), int(np.random.randint(30, 200))
            coco["annotations"].append({"id": ann_id, "image_id": i, "category_id": 1, "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
            ann_id += 1
    (DATA / "annotations.json").write_text(json.dumps(coco))
    (CONFIGS / "data.yaml").write_text(f'img_dir: "{IMGS}"\nann_file: "{DATA / "annotations.json"}"\n')
    print("Synthetic data created.")

## 5 — Train

Progress bar per batch + training snapshot per epoch (progress %, ETA, best loss, GPU mem).

In [ ]:
if USE_REAL_DATA:
    batch_size = BATCH_SIZE
    if batch_size <= 0:
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        batch_size = lt.auto_batch_size(gpu_mem_gb, IMG_SIZE)
        print(f"Auto batch size: {batch_size}")

    model_cfg, data_cfg = lt.write_configs(
        WORKSPACE, data_info, num_classes,
        epochs=EPOCHS, batch_size=batch_size,
        img_size=IMG_SIZE, lr=LR, model_type=MODEL,
    )
    lt.run_training(model_cfg, data_cfg, data_info, DEVICE,
                    epochs=EPOCHS, batch_size=batch_size)
else:
    from badger_vision import Badger_vision
    model = Badger_vision(str(CONFIGS / "resnext_nano.yaml"))
    model.train(data=str(CONFIGS / "data.yaml"), task="detection")

## 6 — Inference & Benchmark

In [ ]:
if not USE_REAL_DATA:
    preds = model.predict(str(IMGS / "img_0000.jpg"))
    for key, val in preds.items():
        print(f"{key}: shape={val.shape}, dtype={val.dtype}")

    results = model.benchmark(warmup=3, runs=20)
    print(f"Latency: {results['avg_latency_ms']} ms  |  FPS: {results['fps']}")
    print(f"Params : {results['params_M']}M  |  FLOPs: {results['flops_G']} G")

    model.export(format="onnx")
    !ls -lh model.onnx
else:
    import glob
    runs = sorted(glob.glob("runs/train_*"))
    if runs:
        best = f"{runs[-1]}/checkpoint_best.pt"
        last = f"{runs[-1]}/checkpoint_last.pt"
        ckpt_path = best if Path(best).exists() else last
        if Path(ckpt_path).exists():
            ckpt = torch.load(ckpt_path, map_location="cpu")
            print(f"Best checkpoint: {ckpt_path}")
            print(f"Epoch: {ckpt.get('epoch', '?')}  |  Loss: {ckpt.get('best_loss', '?'):.4f}")

---

Set `DATASET_PATH` above to your dataset archive or folder and re-run to train on real data.

See the [README](https://github.com/Dillun-Holmes/AI_vision_model) for all model variants and options.